This is the beginning of the project.

In [1]:
'Setting up Imports'
# !pip install ydata-profiling[notebook]
import pandas as pd
from scipy.stats import zscore
from ydata_profiling import ProfileReport
import numpy as np
import re

/Library/Python/3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
CFI = pd.read_csv("./data/Public_CFI_Network_Directory-4_26_2025.csv")
HIP = pd.read_csv("./data/High Impact Professionals - Public Talent Directory-4_26_2025.csv")

HIP.head()

,Full Name,Open to,Cause Areas,Roles: Interested,Roles: 1+ years of experience,Years of Work Experience,Professional Summary / Experience,LinkedIn/CV Link,How interested are you in moving into a (new) high-impact job?,How interested are you in founding/co-founding a new high-impact project?,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Kay Kozaronek,Full-time,"AI strategy and policy,AI technical research,E...","Business Development,Community building,Consul...","Business Development,Administrative,Community ...",4.0,I am a Product leader with experience in AI sa...,https://www.linkedin.com/in/kay-kozaronek/,5 - I’m actively trying to do this,4,...,"Polish,German,English","In-person,Partially remote / Hybrid","Yes,Maybe",I am flexible within Europe + UK and would be ...,Working part time,US$200000,3.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
1,Rebecca Graves,"Full-time,Freelance","Animal welfare,Climate change,Environment and ...","Administrative,Animal Health / Welfare,Animal ...","Accounting,Administrative,Animal Health / Welf...",27.0,"Ambitious animal welfare advocate, successfull...",https://www.linkedin.com/in/becky-graves-21526246,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"English,Spanish",Partially remote / Hybrid,"Maybe,Yes","Prefer to reside in US, but support missions g...",Working full time,US$40000,60.0,"Running a EA workplace or professional group,R...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2,VIRGÍNIA GUARNERI,"Full-time,Part-time",Animal welfare,"Animal Health / Welfare,Animal Science,Busines...",IT,12.0,I'm an IT consultant with 10+ years experience...,https://www.linkedin.com/mwlite/profile/in/vir...,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"Italian,Spanish,English,Portuguese","Fully remote,Partially remote / Hybrid",Yes,I could relocate anywhere but I'm legally allo...,Working full time,NaN,5.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
3,Kiggundu Ronald Reagan,"Volunteering,Part-time","Animal welfare,Environment and sustainability,...","Administrative,Education,Personal Assistant / ...","Administrative,Animal Health / Welfare,Researc...",4.0,"An enthusiastic, self motivated, dedicated, go...",https://www.linkedin.com/in/kiggundu-ronald-re...,5 - I’m actively trying to do this,3 - Not actively looking but would consider a ...,...,English,Fully remote,No,NaN,"Volunteering,Working part time",US$40000,5.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
4,Abubakar Sadiq Umar 3,"Volunteering,Part-time","AI strategy and policy,Animal welfare,Authorit...","Administrative,Animal Health / Welfare,Campaig...","Administrative,Animal Health / Welfare,Campaig...",4.0,Abubakar Sadiq umar is an administrator and a ...,https://www.linkedin.com/in/umar-abubakar-sadi...,5 - I’m actively trying to do this,4,...,English,Partially remote / Hybrid,Yes,"I am willing to stay within Europe, UK and Ame...","Volunteering,Working part time",US$9000,100.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...


In [3]:
CFI.tail()

,Full Name,City,Country,Current Title,Current Employer,Large Firm Affiliation,LinkedIn,Career Path Interests,Cause Area Interests,Openness to a New Role
362,Zachary Thomas,Matawan,United States of America,Operations & HR,EACN,NaN,https://www.linkedin.com/in/zacharythomas10/,"Operations,Founding an organisation,Management...","Building effective altruism,Existential risk r...",NaN
363,Zack Cunich,Canberra,Australia,Junior consultant (0-2 years consulting experi...,NaN,PricewaterhouseCoopers,https://www.linkedin.com/in/zack-cunich-9212b5...,"Policy/Government,Research,Grantmaking,Outreac...","Building effective altruism,Existential risk r...",True
364,Zakaria Bahlouli,Dubai,United Arab Emirates,Independent consultant,Self-Employed,Bain,https://www.linkedin.com/in/zakaria-bahlouli/,"Communications/Marketing,Academic research,Gra...",Undecided,True
365,Tan Zhong Chen,Singapore,Singapore,Junior consultant (0-2 years consulting experi...,The Bridgespan Group,NaN,https://www.linkedin.com/in/tanzhongchen/,"Communications/Marketing,Grantmaking,Operation...","Building effective altruism,Improving institut...",NaN
366,Zacarias Filipe Zandamela,Maputo,Mozambique,Project leader (4-6 years experience),"CSBF, Lda Consulting Firm",NaN,www.linkedin.com/in/zacarias-zandamela-mba-pgd...,"Academic research,EA community building/commun...","Climate change,Poverty and health ( low-income...",True


I don't currently know any details about these datasets. I don't want to prematurely drop any important records, especially when I don't know anything about them. The goal here is that eventually, we will merge the datasets together. First, we will clean each dataset separately, before we merge using concatenation. This will ensure better data integrity, and concatenation will keep all unique entries rather than only the overlapping observations.

In [4]:
# data profiling: check column types, summarize missing values, look at unique values of categorical fields to identify inconsistencies, generate descriptive statistics

def data_profiling(df):
    print("**Basic Data Overview**")
    print(df.info())
    print("\n")
    
    print("**Missing Values Summary**")
    missings = df.isnull().sum()
    print(missings[missings > 0])
    print("\n")
    
    print("**Unique Values in Categorical Columns**")
    categories = df.select_dtypes(include=['object']).columns
    for col in categories:
        unique_values = df[col].unique()
        print(f"{col}: {len(unique_values)} unique values")
        print(f"Sample values: {unique_values[:10]}")
    print("\n")
    
    print("**Descriptive Statistics for Numeric Columns**")
    print(df.describe())

In [5]:
data_profiling(CFI)

**Basic Data Overview**
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Full Name               367 non-null    object
 1   City                    365 non-null    object
 2   Country                 366 non-null    object
 3   Current Title           337 non-null    object
 4   Current Employer        233 non-null    object
 5   Large Firm Affiliation  145 non-null    object
 6   LinkedIn                364 non-null    object
 7   Career Path Interests   324 non-null    object
 8   Cause Area Interests    276 non-null    object
 9   Openness to a New Role  254 non-null    object
dtypes: object(10)
memory usage: 28.8+ KB
None


**Missing Values Summary**
City                        2
Country                     1
Current Title              30
Current Employer          134
Large Firm Affiliation    222
LinkedIn             

In [6]:
data_profiling(HIP)

**Basic Data Overview**
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2579 entries, 0 to 2578
Data columns (total 26 columns):
 #   Column                                                                     Non-Null Count  Dtype  
---  ------                                                                     --------------  -----  
 0   Full Name                                                                  2579 non-null   object 
 1   Open to                                                                    2574 non-null   object 
 2   Cause Areas                                                                2578 non-null   object 
 3   Roles: Interested                                                          2578 non-null   object 
 4   Roles: 1+ years of experience                                              2565 non-null   object 
 5   Years of Work Experience                                                   2572 non-null   float64
 6   Professional Summary / Experienc

Instead of only focusing on manual cleaning, we can also do some data profiling using the package y-data profiling to make a data quality assessment of our datasets before we actually clean them.

In [7]:
CFI_sample = CFI.sample(n=200, random_state=50)
HIP_sample = HIP.sample(frac=0.05, random_state=40)
ydata_quality_report_CFI = ProfileReport(CFI_sample, title='Pre-Cleaning CFI Quality Report')
ydata_quality_report_HIP = ProfileReport(HIP_sample, title='Pre-Cleaning HIP Quality Report', minimal=True)

In [8]:
ydata_quality_report_CFI.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 10/10 [00:00<00:00, 261.00it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
ydata_quality_report_HIP.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 26/26 [00:00<00:00, 112.53it/s]A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
# may be too big to view within an IDE
ydata_quality_report_HIP.to_file("Pre-Cleaning HIP Quality Report.html")

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
# syntactic and pattern based error detection/repair on multi-value columns

def standardization(df, column_name):
    df[column_name] = df[column_name].astype(str).str.strip()
    df[column_name] = df[column_name].str.replace(r'\s*,\s*', ',', regex=True) #replace whitespace around a comma
    df[column_name] = df[column_name].str.replace(r'\s*/\s*', '/', regex=True) # replace whitespace around a slash
    return df

In [12]:
standardization(CFI, "Cause Area Interests")
standardization(CFI, "Career Path Interests")
standardization(HIP, "Cause Areas")
standardization(HIP, "Open to")
standardization(HIP, "Roles: Interested")
standardization(HIP, "Roles: 1+ years of experience")
standardization(HIP, "New Project - Needs")
standardization(HIP, "Languages")
standardization(HIP, "Work Location Preference")
standardization(HIP, "Current Employment Status")
standardization(HIP, "Other Interests")

,Full Name,Open to,Cause Areas,Roles: Interested,Roles: 1+ years of experience,Years of Work Experience,Professional Summary / Experience,LinkedIn/CV Link,How interested are you in moving into a (new) high-impact job?,How interested are you in founding/co-founding a new high-impact project?,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Kay Kozaronek,Full-time,"AI strategy and policy,AI technical research,E...","Business Development,Community building,Consul...","Business Development,Administrative,Community ...",4.0,I am a Product leader with experience in AI sa...,https://www.linkedin.com/in/kay-kozaronek/,5 - I’m actively trying to do this,4,...,"Polish,German,English","In-person,Partially remote/Hybrid","Yes,Maybe",I am flexible within Europe + UK and would be ...,Working part time,US$200000,3.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
1,Rebecca Graves,"Full-time,Freelance","Animal welfare,Climate change,Environment and ...","Administrative,Animal Health/Welfare,Animal Sc...","Accounting,Administrative,Animal Health/Welfar...",27.0,"Ambitious animal welfare advocate, successfull...",https://www.linkedin.com/in/becky-graves-21526246,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"English,Spanish",Partially remote/Hybrid,"Maybe,Yes","Prefer to reside in US, but support missions g...",Working full time,US$40000,60.0,"Running a EA workplace or professional group,R...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2,VIRGÍNIA GUARNERI,"Full-time,Part-time",Animal welfare,"Animal Health/Welfare,Animal Science,Business ...",IT,12.0,I'm an IT consultant with 10+ years experience...,https://www.linkedin.com/mwlite/profile/in/vir...,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"Italian,Spanish,English,Portuguese","Fully remote,Partially remote/Hybrid",Yes,I could relocate anywhere but I'm legally allo...,Working full time,NaN,5.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
3,Kiggundu Ronald Reagan,"Volunteering,Part-time","Animal welfare,Environment and sustainability,...","Administrative,Education,Personal Assistant/Ex...","Administrative,Animal Health/Welfare,Research,...",4.0,"An enthusiastic, self motivated, dedicated, go...",https://www.linkedin.com/in/kiggundu-ronald-re...,5 - I’m actively trying to do this,3 - Not actively looking but would consider a ...,...,English,Fully remote,No,NaN,"Volunteering,Working part time",US$40000,5.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
4,Abubakar Sadiq Umar 3,"Volunteering,Part-time","AI strategy and policy,Animal welfare,Authorit...","Administrative,Animal Health/Welfare,Campaigns...","Administrative,Animal Health/Welfare,Campaigns...",4.0,Abubakar Sadiq umar is an administrator and a ...,https://www.linkedin.com/in/umar-abubakar-sadi...,5 - I’m actively trying to do this,4,...,English,Partially remote/Hybrid,Yes,"I am willing to stay within Europe, UK and Ame...","Volunteering,Working part time",US$9000,100.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,Greg Gianopoulos,"Full-time,Part-time","EA movement building/growth,Education and empl...","Education,Community building","Communications,Community building",9.0,Created Charity Elections program supported by...,https://www.linkedin.com/in/greggianopoulos/,2,1 - I’m not interested in changing roles right...,...,English,Fully remote,No,NaN,"Studying,Working part time",US$110100,7.0,Running a EA workplace or professional group,2025-01-07,https://airtable

In [13]:
HIP['Country'].value_counts()

Country
United States of America           723
United Kingdom                     325
Germany                            128
India                              117
Netherlands                         97
                                  ... 
Ireland,United Kingdom,Pakistan      1
United States of America,France      1
Germany,United Kingdom               1
Australia,Canada                     1
United Kingdom,Indonesia,Italy       1
Name: count, Length: 181, dtype: int64

In [14]:
CFI['Country'].value_counts()

Country
United States of America    111
United Kingdom               67
Germany                      30
Canada                       13
India                        13
Netherlands                  12
Australia                    11
Kenya                         7
Norway                        6
Austria                       6
France                        6
Sweden                        5
Brazil                        4
Poland                        4
Belgium                       4
Czechia (Czech Republic)      4
Singapore                     3
Switzerland                   3
United Arab Emirates          3
Philippines                   3
Vietnam                       3
Spain                         3
Nigeria                       2
Finland                       2
Portugal                      2
South Africa                  2
Peru                          2
Hungary                       2
Argentina                     2
Israel                        2
Cambodia                      2


In [15]:
def punctuation_elimination(df, column_name):
    df[column_name] = df[column_name].str.replace(r'[.,/()\-&"]', '', regex=True)
    df[column_name] = df[column_name].str.title()
    return df

In [16]:
punctuation_elimination(CFI, "City")
punctuation_elimination(CFI, "Country")
punctuation_elimination(HIP, "City")
punctuation_elimination(HIP, "Country")

,Full Name,Open to,Cause Areas,Roles: Interested,Roles: 1+ years of experience,Years of Work Experience,Professional Summary / Experience,LinkedIn/CV Link,How interested are you in moving into a (new) high-impact job?,How interested are you in founding/co-founding a new high-impact project?,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Kay Kozaronek,Full-time,"AI strategy and policy,AI technical research,E...","Business Development,Community building,Consul...","Business Development,Administrative,Community ...",4.0,I am a Product leader with experience in AI sa...,https://www.linkedin.com/in/kay-kozaronek/,5 - I’m actively trying to do this,4,...,"Polish,German,English","In-person,Partially remote/Hybrid","Yes,Maybe",I am flexible within Europe + UK and would be ...,Working part time,US$200000,3.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
1,Rebecca Graves,"Full-time,Freelance","Animal welfare,Climate change,Environment and ...","Administrative,Animal Health/Welfare,Animal Sc...","Accounting,Administrative,Animal Health/Welfar...",27.0,"Ambitious animal welfare advocate, successfull...",https://www.linkedin.com/in/becky-graves-21526246,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"English,Spanish",Partially remote/Hybrid,"Maybe,Yes","Prefer to reside in US, but support missions g...",Working full time,US$40000,60.0,"Running a EA workplace or professional group,R...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2,VIRGÍNIA GUARNERI,"Full-time,Part-time",Animal welfare,"Animal Health/Welfare,Animal Science,Business ...",IT,12.0,I'm an IT consultant with 10+ years experience...,https://www.linkedin.com/mwlite/profile/in/vir...,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"Italian,Spanish,English,Portuguese","Fully remote,Partially remote/Hybrid",Yes,I could relocate anywhere but I'm legally allo...,Working full time,NaN,5.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
3,Kiggundu Ronald Reagan,"Volunteering,Part-time","Animal welfare,Environment and sustainability,...","Administrative,Education,Personal Assistant/Ex...","Administrative,Animal Health/Welfare,Research,...",4.0,"An enthusiastic, self motivated, dedicated, go...",https://www.linkedin.com/in/kiggundu-ronald-re...,5 - I’m actively trying to do this,3 - Not actively looking but would consider a ...,...,English,Fully remote,No,NaN,"Volunteering,Working part time",US$40000,5.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
4,Abubakar Sadiq Umar 3,"Volunteering,Part-time","AI strategy and policy,Animal welfare,Authorit...","Administrative,Animal Health/Welfare,Campaigns...","Administrative,Animal Health/Welfare,Campaigns...",4.0,Abubakar Sadiq umar is an administrator and a ...,https://www.linkedin.com/in/umar-abubakar-sadi...,5 - I’m actively trying to do this,4,...,English,Partially remote/Hybrid,Yes,"I am willing to stay within Europe, UK and Ame...","Volunteering,Working part time",US$9000,100.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,Greg Gianopoulos,"Full-time,Part-time","EA movement building/growth,Education and empl...","Education,Community building","Communications,Community building",9.0,Created Charity Elections program supported by...,https://www.linkedin.com/in/greggianopoulos/,2,1 - I’m not interested in changing roles right...,...,English,Fully remote,No,NaN,"Studying,Working part time",US$110100,7.0,Running a EA workplace or professional group,2025-01-07,https://airtable

I've decided I only want to see people's LinkedIn profiles, not resumes, from the HIP data, due potential invalid URL formats and to make the columns similar. 

In [17]:
print(len(HIP))

2579


In [18]:
HIP = HIP[HIP['LinkedIn/CV Link'].str.contains(r'(https://www\.linkedin\.com|www\.linkedin\.com)', na=False)]

/var/folders/vx/f0n6zqf15h9c22pcp057txxw0000gn/T/ipykernel_14533/530919668.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  HIP = HIP[HIP['LinkedIn/CV Link'].str.contains(r'(https://www\.linkedin\.com|www\.linkedin\.com)', na=False)]


In [19]:
print(len(HIP))

2141


In [20]:
#standardizing column names to match CFI from HIP.
HIP = HIP.rename(columns={'Cause Areas': 'Cause Area Interests'})
HIP = HIP.rename(columns={'LinkedIn/CV Link': 'LinkedIn'})
HIP.head()

,Full Name,Open to,Cause Area Interests,Roles: Interested,Roles: 1+ years of experience,Years of Work Experience,Professional Summary / Experience,LinkedIn,How interested are you in moving into a (new) high-impact job?,How interested are you in founding/co-founding a new high-impact project?,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Kay Kozaronek,Full-time,"AI strategy and policy,AI technical research,E...","Business Development,Community building,Consul...","Business Development,Administrative,Community ...",4.0,I am a Product leader with experience in AI sa...,https://www.linkedin.com/in/kay-kozaronek/,5 - I’m actively trying to do this,4,...,"Polish,German,English","In-person,Partially remote/Hybrid","Yes,Maybe",I am flexible within Europe + UK and would be ...,Working part time,US$200000,3.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
1,Rebecca Graves,"Full-time,Freelance","Animal welfare,Climate change,Environment and ...","Administrative,Animal Health/Welfare,Animal Sc...","Accounting,Administrative,Animal Health/Welfar...",27.0,"Ambitious animal welfare advocate, successfull...",https://www.linkedin.com/in/becky-graves-21526246,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"English,Spanish",Partially remote/Hybrid,"Maybe,Yes","Prefer to reside in US, but support missions g...",Working full time,US$40000,60.0,"Running a EA workplace or professional group,R...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2,VIRGÍNIA GUARNERI,"Full-time,Part-time",Animal welfare,"Animal Health/Welfare,Animal Science,Business ...",IT,12.0,I'm an IT consultant with 10+ years experience...,https://www.linkedin.com/mwlite/profile/in/vir...,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"Italian,Spanish,English,Portuguese","Fully remote,Partially remote/Hybrid",Yes,I could relocate anywhere but I'm legally allo...,Working full time,NaN,5.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
3,Kiggundu Ronald Reagan,"Volunteering,Part-time","Animal welfare,Environment and sustainability,...","Administrative,Education,Personal Assistant/Ex...","Administrative,Animal Health/Welfare,Research,...",4.0,"An enthusiastic, self motivated, dedicated, go...",https://www.linkedin.com/in/kiggundu-ronald-re...,5 - I’m actively trying to do this,3 - Not actively looking but would consider a ...,...,English,Fully remote,No,NaN,"Volunteering,Working part time",US$40000,5.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
4,Abubakar Sadiq Umar 3,"Volunteering,Part-time","AI strategy and policy,Animal welfare,Authorit...","Administrative,Animal Health/Welfare,Campaigns...","Administrative,Animal Health/Welfare,Campaigns...",4.0,Abubakar Sadiq umar is an administrator and a ...,https://www.linkedin.com/in/umar-abubakar-sadi...,5 - I’m actively trying to do this,4,...,English,Partially remote/Hybrid,Yes,"I am willing to stay within Europe, UK and Ame...","Volunteering,Working part time",US$9000,100.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...


In [21]:
print(len(CFI))
print(len(HIP))

367
2141


In [22]:
#remove exact duplicates within each dataset separately with dr.drop_duplicates
CFI.drop_duplicates(subset=["Full Name", "LinkedIn"], keep='first')
HIP.drop_duplicates(subset=["Full Name", "LinkedIn"], keep="first")

,Full Name,Open to,Cause Area Interests,Roles: Interested,Roles: 1+ years of experience,Years of Work Experience,Professional Summary / Experience,LinkedIn,How interested are you in moving into a (new) high-impact job?,How interested are you in founding/co-founding a new high-impact project?,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Kay Kozaronek,Full-time,"AI strategy and policy,AI technical research,E...","Business Development,Community building,Consul...","Business Development,Administrative,Community ...",4.0,I am a Product leader with experience in AI sa...,https://www.linkedin.com/in/kay-kozaronek/,5 - I’m actively trying to do this,4,...,"Polish,German,English","In-person,Partially remote/Hybrid","Yes,Maybe",I am flexible within Europe + UK and would be ...,Working part time,US$200000,3.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
1,Rebecca Graves,"Full-time,Freelance","Animal welfare,Climate change,Environment and ...","Administrative,Animal Health/Welfare,Animal Sc...","Accounting,Administrative,Animal Health/Welfar...",27.0,"Ambitious animal welfare advocate, successfull...",https://www.linkedin.com/in/becky-graves-21526246,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"English,Spanish",Partially remote/Hybrid,"Maybe,Yes","Prefer to reside in US, but support missions g...",Working full time,US$40000,60.0,"Running a EA workplace or professional group,R...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2,VIRGÍNIA GUARNERI,"Full-time,Part-time",Animal welfare,"Animal Health/Welfare,Animal Science,Business ...",IT,12.0,I'm an IT consultant with 10+ years experience...,https://www.linkedin.com/mwlite/profile/in/vir...,5 - I’m actively trying to do this,5 - I’m actively trying to do this,...,"Italian,Spanish,English,Portuguese","Fully remote,Partially remote/Hybrid",Yes,I could relocate anywhere but I'm legally allo...,Working full time,NaN,5.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
3,Kiggundu Ronald Reagan,"Volunteering,Part-time","Animal welfare,Environment and sustainability,...","Administrative,Education,Personal Assistant/Ex...","Administrative,Animal Health/Welfare,Research,...",4.0,"An enthusiastic, self motivated, dedicated, go...",https://www.linkedin.com/in/kiggundu-ronald-re...,5 - I’m actively trying to do this,3 - Not actively looking but would consider a ...,...,English,Fully remote,No,NaN,"Volunteering,Working part time",US$40000,5.0,Board Member - Sitting on the board of an effe...,2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
4,Abubakar Sadiq Umar 3,"Volunteering,Part-time","AI strategy and policy,Animal welfare,Authorit...","Administrative,Animal Health/Welfare,Campaigns...","Administrative,Animal Health/Welfare,Campaigns...",4.0,Abubakar Sadiq umar is an administrator and a ...,https://www.linkedin.com/in/umar-abubakar-sadi...,5 - I’m actively trying to do this,4,...,English,Partially remote/Hybrid,Yes,"I am willing to stay within Europe, UK and Ame...","Volunteering,Working part time",US$9000,100.0,"Advisor - Advising an effective organization,B...",2025-04-26,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,Greg Gianopoulos,"Full-time,Part-time","EA movement building/growth,Education and empl...","Education,Community building","Communications,Community building",9.0,Created Charity Elections program supported by...,https://www.linkedin.com/in/greggianopoulos/,2,1 - I’m not interested in changing roles right...,...,English,Fully remote,No,NaN,"Studying,Working part time",US$110100,7.0,Running a EA workplace or professional group,2025-01-07,https://airtabl

In [23]:
print(len(CFI))
print(len(HIP))

367
2141


As we can see, xxx.

In [24]:
# handle missing values after some of the preprocessing we just did
null_sums = pd.concat([
    HIP.isnull().sum().rename("HIP_Missing"),
    CFI.isnull().sum().rename("CFI_Missing")
], axis=1)

null_sums = null_sums.fillna(0).astype(int) # they wont have the exact same columns, so for readability we put 0's. 
print(null_sums)

                                                    HIP_Missing  CFI_Missing
Full Name                                                     0            0
Open to                                                       0            0
Cause Area Interests                                          0            0
Roles: Interested                                             0            0
Roles: 1+ years of experience                                 0            0
Years of Work Experience                                      4            0
Professional Summary / Experience                           113            0
LinkedIn                                                      0            3
How interested are you in moving into a (new) h...            0            0
How interested are you in founding/co-founding ...          331            0
New Project - Needs                                           0            0
Time spent engaging with Effective Altruism (EA)              4            0

What gaps and trends exist in career area interest, skill evolution, and global representation across the 2 nonprofit professional networks focused on impact-driven work, and how does data quality impact these insights? 

In [25]:
def missing_handling(df, *column_name, fill_value="Unknown"):
    for col in column_name:
        if col in df.columns:
            df[col] = df[col].fillna(fill_value)
        else:
            print(f"Column '{col}' not found. skipped." )
    return df

In [26]:
CFI = missing_handling(CFI, 
                       "City", "Country", "LinkedIn", "Current Title", "Current Employer", "Large Firm Affiliation", "Openness to a New Role", "Career Path Interests", "Cause Area Interests",
                       fill_value="Unknown")
HIP = missing_handling(HIP, 
                       "Years of Work Experience", "Professional Summary / Experience", "How interested are you in founding/co-founding a new high-impact project?", "Available From", "City", "Country", "Timezone", "Willing to Relocate", "Willing to Relocate - Context", fill_value="Unknown")
HIP = missing_handling(HIP, "Time spent engaging with Effective Altruism (EA)", fill_value="0h")
HIP = missing_handling(HIP, "Money Managed", fill_value="US$0")
HIP = missing_handling(HIP, "Max Staff Managed", fill_value="0.0")

In [27]:
null_sums = pd.concat([
    HIP.isnull().sum().rename("HIP_Missing"),
    CFI.isnull().sum().rename("CFI_Missing")
], axis=1)

null_sums = null_sums.fillna(0).astype(int) 
print(null_sums)

                                                    HIP_Missing  CFI_Missing
Full Name                                                     0            0
Open to                                                       0            0
Cause Area Interests                                          0            0
Roles: Interested                                             0            0
Roles: 1+ years of experience                                 0            0
Years of Work Experience                                      0            0
Professional Summary / Experience                             0            0
LinkedIn                                                      0            0
How interested are you in moving into a (new) h...            0            0
How interested are you in founding/co-founding ...            0            0
New Project - Needs                                           0            0
Time spent engaging with Effective Altruism (EA)              0            0

In [28]:
HIP.dtypes

Full Name                                                                    object
Open to                                                                      object
Cause Area Interests                                                         object
Roles: Interested                                                            object
Roles: 1+ years of experience                                                object
Years of Work Experience                                                     object
Professional Summary / Experience                                            object
LinkedIn                                                                     object
How interested are you in moving into a (new) high-impact job?               object
How interested are you in founding/co-founding a new high-impact project?    object
New Project - Needs                                                          object
Time spent engaging with Effective Altruism (EA)                            

In [29]:
# need to convert appropriate string columns to numeric, to detect outliers
def numeric_conversions(df):
    if "Time spent engaging with Effective Altruism (EA)" in df.columns:
        df["Time spent engaging with Effective Altruism (EA)"] = df["Time spent engaging with Effective Altruism (EA)"].astype(str).str.replace("h", "", regex=False)
        df["Time spent engaging with Effective Altruism (EA)"] = pd.to_numeric(df["Time spent engaging with Effective Altruism (EA)"], errors='coerce')

    if "Money Managed" in df.columns:
        df["Money Managed"] = df["Money Managed"].astype(str).str.replace("US$", "", regex=False)
        df["Money Managed"] = pd.to_numeric(df["Money Managed"], errors='coerce')
    
    if "Max Staff Managed" in df.columns:
        df['Max Staff Managed'] = df["Max Staff Managed"].astype(str)
        df["Max Staff Managed"] = df["Max Staff Managed"].str.replace(r'[^\d.]', '', regex=True)
        df['Max Staff Managed'] = pd.to_numeric(df['Max Staff Managed'], errors='coerce')
    
    return df

In [30]:
HIP = numeric_conversions(HIP)

In [31]:
# handle outliers
# z scores for unrealistic values & repair the values : Years of Work Experience, Money Managed, Max Staff Managed
def detecting_numeric_outliers(df, columns, threshold=3):
    specific_outliers = []
    for col in columns:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        not_null_vals = df[col].dropna()
        zscores = zscore(not_null_vals)
        outlier_ids = not_null_vals[abs(zscores) > threshold]
    for idx, value in outlier_ids.items():
        specific_outliers.append({
            "Index": idx,
            "Outlier Column": col,
            "Outlier Value": value,
            **{col: value}
        })
    return  pd.DataFrame(specific_outliers)

In [32]:
numeric_columns = ["Time spent engaging with Effective Altruism (EA)", "Money Managed", "Max Staff Managed"]
outliers = detecting_numeric_outliers(HIP, numeric_columns, threshold=3)
print(f"Outliers detected: {len(outliers)}")
display(outliers)

Outliers detected: 4


,Index,Outlier Column,Outlier Value,Max Staff Managed
0,248,Max Staff Managed,480000.0,480000.0
1,2011,Max Staff Managed,670000.0,670000.0
2,2112,Max Staff Managed,200000.0,200000.0
3,2144,Max Staff Managed,300000.0,300000.0


In [33]:
def repair_outliers(df, outliers_detected, replacement_val = "0.0"):
    for _, row in outliers_detected.iterrows():
        index = row["Index"]
        column = row["Outlier Column"]
        if column in df.columns and index in df.index:
            df.at[index, column] = replacement_val
    return df

In [34]:
numeric_columns = ["Time spent engaging with Effective Altruism (EA)", "Money Managed", "Max Staff Managed"]
outliers = detecting_numeric_outliers(HIP, numeric_columns, threshold=3)
HIP = repair_outliers(HIP, outliers, replacement_val="0")

/var/folders/vx/f0n6zqf15h9c22pcp057txxw0000gn/T/ipykernel_14533/1306370454.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, column] = replacement_val


Merge point.

In [35]:
combined_data = pd.concat([CFI, HIP], axis=0, ignore_index=True, sort=False)
combined_data = combined_data.fillna("Not Available for Both Datasets")

combined_data

,Full Name,City,Country,Current Title,Current Employer,Large Firm Affiliation,LinkedIn,Career Path Interests,Cause Area Interests,Openness to a New Role,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Abdullah Al Qayyim,New Brunswick,Canada,Independent consultant,"Abdullahandbeyond, MCAF",Unknown,https://www.linkedin.com/in/abdullahandbeyond/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
1,Aditi Sharma,Bangalore,India,Consultant (2-4 years experience),VMware,Unknown,https://www.linkedin.com/in/aditishm/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
2,Andrew Dornon,Austin,United States Of America,Other,TaskHuman,Unknown,https://www.linkedin.com/in/andrewdornon/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
3,Anurag Dey,Stockholm,Sweden,Consultant (2-4 years experience),Analysys Mason,Unknown,https://www.linkedin.com/in/anuragdey/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
4,Barth Cabouat,Nairobi,Kenya,Other,One Acre Fund,Unknown,https://www.linkedin.com/in/barthcabouat/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2503,Greg Gianopoulos,Asheville North Carolina,United States Of America,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/greggianopoulos/,Not Available for Both Datasets,"EA movement building/growth,Education and empl...",Not Available for Both Datasets,...,English,Fully remote,No,Unknown,"Studying,Working part time",110100.0,7.0,Running a EA workplace or professional group,2025-01-07,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2504,Richard Annilo,Tartu,Estonia,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/richardannilo/,Not Available for Both Datasets,"AI strategy and policy,AI technical research,A...",Not Available for Both Datasets,...,"Estonian,English","Fully remote,Partially remote/Hybrid,In-person",Yes,Unknown,Freelancing,100000.0,5.0,Board Member - Sitting on the board of an effe...,2025-02-23,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2505,Jack Malde,Washington Dc,United States Of America,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/jack-malde/,Not Available for Both Datasets,"AI str

In [36]:
# identify and handle post merge duplicates 
are_there_duplicates = combined_data[combined_data.duplicated(subset=["Full Name", "LinkedIn"], keep=False)]
display(are_there_duplicates)

,Full Name,City,Country,Current Title,Current Employer,Large Firm Affiliation,LinkedIn,Career Path Interests,Cause Area Interests,Openness to a New Role,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
23,Luke Eure,Nairobi,Kenya,Other,Kapu,Unknown,https://www.linkedin.com/in/luke-eure-177015151/,Outreach & Community Building,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
30,Nils Voelker,Hamburg,Germany,Other,Unknown,Accenture,https://www.linkedin.com/in/nils-voelker/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
40,Kent Lui,Chicago,United States Of America,Unknown,Unknown,Unknown,https://www.linkedin.com/in/kentluikentlui/,"Finance/Accounting,HR/People operations,Produc...",nan,True,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
45,Ada Sevimli,Berlin,Germany,Consultant (2-4 years experience),BTO Management Consulting AG,Unknown,https://www.linkedin.com/in/ada-sevimli-4b417b...,"Communications/Marketing,Grantmaking,EA commun...","Animal welfare,Climate change,Environment and ...",True,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
52,Andreas Pashos,New Orleans,United States Of America,Independent consultant,Unknown,Unknown,https://www.linkedin.com/in/andreaspashos/,"Management,Operations,Founding an organisation...",nan,True,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2469,Faiz Jamdar,Melbourne,Australia,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/faizjamdar/,Not Available for Both Datasets,"Economic inequality,Education and employment,M...",Not Available for Both Datasets,...,English,Fully remote,No,Unknown,Working full time,50000.0,0.0,"Advisor - Advising an effective organization,P...",2025-04-10,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2471,Nils Voelker,Hamburg,Germany,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/nils-voelker/,Not Available for Both Datasets,Global health and development,Not Available for Both Datasets,...,"German,English",nan,Unknown,Unknown,Working full time,300000.0,5.0,"Advisor - Advising an effective organization,B...",2025-03-01,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2473,Sanjana Kashyap,Mumbai,India,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,http

In [37]:
combined_data = combined_data.drop_duplicates(subset=["Full Name", "LinkedIn"], keep="first")

In [38]:
combined_data

,Full Name,City,Country,Current Title,Current Employer,Large Firm Affiliation,LinkedIn,Career Path Interests,Cause Area Interests,Openness to a New Role,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Abdullah Al Qayyim,New Brunswick,Canada,Independent consultant,"Abdullahandbeyond, MCAF",Unknown,https://www.linkedin.com/in/abdullahandbeyond/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
1,Aditi Sharma,Bangalore,India,Consultant (2-4 years experience),VMware,Unknown,https://www.linkedin.com/in/aditishm/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
2,Andrew Dornon,Austin,United States Of America,Other,TaskHuman,Unknown,https://www.linkedin.com/in/andrewdornon/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
3,Anurag Dey,Stockholm,Sweden,Consultant (2-4 years experience),Analysys Mason,Unknown,https://www.linkedin.com/in/anuragdey/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
4,Barth Cabouat,Nairobi,Kenya,Other,One Acre Fund,Unknown,https://www.linkedin.com/in/barthcabouat/,nan,nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2502,Nayanika Kundu,Unknown,India,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/nayanika-k,Not Available for Both Datasets,"Animal welfare,EA movement building/growth,Edu...",Not Available for Both Datasets,...,"English,Hindi,Bengali","Fully remote,Partially remote/Hybrid,In-person",Yes,I would need a visa sponsorship,"Volunteering,Freelancing",0.0,0.0,Advisor - Advising an effective organization,2025-01-07,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2503,Greg Gianopoulos,Asheville North Carolina,United States Of America,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/greggianopoulos/,Not Available for Both Datasets,"EA movement building/growth,Education and empl...",Not Available for Both Datasets,...,English,Fully remote,No,Unknown,"Studying,Working part time",110100.0,7.0,Running a EA workplace or professional group,2025-01-07,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2504,Richard Annilo,Tartu,Estonia,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/richardannilo/,Not Available for Both Data

In [39]:
# align and standardize final categorical fields if applicable
def categorical_standards(df, columns):
    for col in columns:
        if col in df.columns:
            df.loc[:, col] = df[col].astype(str).str.strip().str.title()
    return df

In [40]:
cat_cols = ["City", "Country", "Career Path Interests", "Cause Area Interests"]
categorical_standards(combined_data, cat_cols)

,Full Name,City,Country,Current Title,Current Employer,Large Firm Affiliation,LinkedIn,Career Path Interests,Cause Area Interests,Openness to a New Role,...,Languages,Work Location Preference,Willing to Relocate,Willing to Relocate - Context,Current Employment Status,Money Managed,Max Staff Managed,Other Interests,Profile Last Updated,Outdated Candidate
0,Abdullah Al Qayyim,New Brunswick,Canada,Independent consultant,"Abdullahandbeyond, MCAF",Unknown,https://www.linkedin.com/in/abdullahandbeyond/,Nan,Nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
1,Aditi Sharma,Bangalore,India,Consultant (2-4 years experience),VMware,Unknown,https://www.linkedin.com/in/aditishm/,Nan,Nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
2,Andrew Dornon,Austin,United States Of America,Other,TaskHuman,Unknown,https://www.linkedin.com/in/andrewdornon/,Nan,Nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
3,Anurag Dey,Stockholm,Sweden,Consultant (2-4 years experience),Analysys Mason,Unknown,https://www.linkedin.com/in/anuragdey/,Nan,Nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
4,Barth Cabouat,Nairobi,Kenya,Other,One Acre Fund,Unknown,https://www.linkedin.com/in/barthcabouat/,Nan,Nan,Unknown,...,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2502,Nayanika Kundu,Unknown,India,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/nayanika-k,Not Available For Both Datasets,"Animal Welfare,Ea Movement Building/Growth,Edu...",Not Available for Both Datasets,...,"English,Hindi,Bengali","Fully remote,Partially remote/Hybrid,In-person",Yes,I would need a visa sponsorship,"Volunteering,Freelancing",0.0,0.0,Advisor - Advising an effective organization,2025-01-07,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2503,Greg Gianopoulos,Asheville North Carolina,United States Of America,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/greggianopoulos/,Not Available For Both Datasets,"Ea Movement Building/Growth,Education And Empl...",Not Available for Both Datasets,...,English,Fully remote,No,Unknown,"Studying,Working part time",110100.0,7.0,Running a EA workplace or professional group,2025-01-07,https://airtable.com/shrSvHwSxxAOAPDxO?prefill...
2504,Richard Annilo,Tartu,Estonia,Not Available for Both Datasets,Not Available for Both Datasets,Not Available for Both Datasets,https://www.linkedin.com/in/richardannilo/,Not Available For Both Data

In [41]:
# final missing values check
print(combined_data.isnull().sum())

Full Name                                                                    0
City                                                                         0
Country                                                                      0
Current Title                                                                0
Current Employer                                                             0
Large Firm Affiliation                                                       0
LinkedIn                                                                     0
Career Path Interests                                                        0
Cause Area Interests                                                         0
Openness to a New Role                                                       0
Open to                                                                      0
Roles: Interested                                                            0
Roles: 1+ years of experience                       

In [42]:
combined_data = combined_data.replace(["nan", "NaN"], np.nan)
print(combined_data.isnull().sum())

Full Name                                                                      0
City                                                                           0
Country                                                                        0
Current Title                                                                  0
Current Employer                                                               0
Large Firm Affiliation                                                         0
LinkedIn                                                                       0
Career Path Interests                                                          0
Cause Area Interests                                                           0
Openness to a New Role                                                         0
Open to                                                                        4
Roles: Interested                                                              0
Roles: 1+ years of experienc

In [43]:
def final_missing(df, fill_value="Something"):
    return df.fillna(fill_value)

In [44]:
combined_data = final_missing(combined_data, fill_value="Missing Information")

In [46]:
#write the csv, end part 1.
combined_data.to_csv("./data/combined_data_cleaned.csv", index=False)